In [1]:
import ee
import geemap
import pandas as pd

ee.Initialize(project='ee-chowdharypranav2005')

print("Earth Engine Connected!")

Earth Engine Connected!


In [2]:
forest = ee.Image(
    "UMD/hansen/global_forest_change_2025_v1_13"
)

In [3]:
print(forest.bandNames().getInfo())

['treecover2000', 'loss', 'gain', 'lossyear', 'first_b30', 'first_b40', 'first_b50', 'first_b70', 'last_b30', 'last_b40', 'last_b50', 'last_b70', 'datamask']


In [4]:
loss_year = forest.select("lossyear")

print("Loss year band selected!")

Loss year band selected!


In [5]:
Map = geemap.Map()

vis_params = {
    "min": 1,
    "max": 24,
    "palette": ["yellow", "orange", "red"]
}

Map.addLayer(loss_year, vis_params, "Forest Loss")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [6]:
region = ee.Geometry.Point([73.7898, 19.9975]).buffer(50000)

loss_stats = loss_year.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=region,
    scale=30,
    maxPixels=1e9
)

print(loss_stats.getInfo())

{'lossyear': 15.15484927916121}


In [7]:
forest_loss_df = pd.DataFrame([loss_stats.getInfo()])

forest_loss_df.head()

,lossyear
0,15.154849


In [8]:
loss = forest.select("lossyear")

loss_mask = loss.gt(0)

print("Forest loss mask created!")

Forest loss mask created!


In [9]:
region = ee.Geometry.Point([73.7898, 19.9975]).buffer(50000)

In [10]:
forest_loss_area = loss_mask.multiply(ee.Image.pixelArea())

stats = forest_loss_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=region,
    scale=30,
    maxPixels=1e9
)

loss_area_m2 = stats.getInfo()["lossyear"]

loss_area_km2 = loss_area_m2 / 1e6

print("Forest Loss Area (km²):", loss_area_km2)

Forest Loss Area (km²): 0.4532643365064434


In [11]:
import pandas as pd

forest_df = pd.DataFrame({
    "forest_loss_area_km2": [loss_area_km2]
})

forest_df.to_csv(
    "../data/processed/forest_loss_stats.csv",
    index=False
)

print("Forest loss stats saved!")

Forest loss stats saved!
